# Basic OPET Operations

This notebook demonstrates core operations with the OPET photovoltaic electronic load:

1. Connecting to an OPET over RS485
2. Reading device info and status
3. Running MPPT and collecting point measurements
4. Capturing and plotting I-V curves
5. Sending raw commands

In [ ]:
from OPET_control import OPETBus, OPET
from serial import Serial
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

## Connect to the OPET

Edit the serial port and address below to match your setup:
- **Port:** On macOS/Linux, look for `/dev/tty*` or `/dev/cu.*` devices. On Windows, use something like `'COM3'`.
- **Address:** Set with jumpers on the OPET board (0–31).

In [ ]:
# ---- Edit these to match your setup ----
opet_port_name = '/dev/ttyUSB0'  # serial port connected to the OPET bus
opet_address = 5                 # jumper-configured address (0-31)
# -----------------------------------------

opet_port = Serial(
    opet_port_name,
    baudrate=200000,
    timeout=1
)

opet_bus = OPETBus(opet_port)
opet = OPET(opet_bus, opet_address)

## Device info and status

In [ ]:
# Get identifying information from the OPET
opet.identification

In [ ]:
# Read the status
opet.status

## Point measurements

Activate MPPT mode and collect a series of single-point measurements.

In [ ]:
# Activate MPPT
opet.mode = 'mppt'  # 'isc' and 'voc' are also available
opet.output_enabled = True

In [ ]:
# Collect a series of point measurements
samples = [opet.sample for _ in range(200)]

In [ ]:
# Plot the point measurement results
fig, ax_array = plt.subplots(nrows=3, sharex=True)

ax = ax_array[0]
ax.plot(
    [x['measurement_time'] for x in samples],
    [x['voltage'] for x in samples]
)
ax.set_ylabel('voltage / V')

ax = ax_array[1]
ax.plot(
    [x['measurement_time'] for x in samples],
    [x['current'] for x in samples]
)
ax.set_ylabel('current / A')

ax = ax_array[2]
ax.plot(
    [x['measurement_time'] for x in samples],
    [x['power'] for x in samples]
)
ax.set_ylabel('power / W')
plt.xticks(rotation=-10)
pass

## I-V curves

Request an I-V curve, wait for it to finish, then plot the result. The second example uses `send_verify` to change the number of IV points before re-measuring.

In [ ]:
# Request an IV curve
opet.start_iv_curve()

# Wait for it to complete
while not opet.available:
    pass

# Read out the result
iv_result = opet.iv_data

In [ ]:
# Plot the IV curve
fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.plot(iv_result['voltage'], iv_result['current'])
ax.set_xlabel('voltage / V')
ax.set_ylabel('current / A')
pass

### Changing IV curve settings with a raw command

Any command from the OPET manual can be sent using `send_verify`. Here we change the number of I-V curve points to 200, then re-measure.

In [ ]:
opet.send_verify('IV:POINTS\t200')

In [ ]:
# Request an IV curve
opet.start_iv_curve()

# Wait for it to complete
while not opet.available:
    pass

# Read out the result
iv_result = opet.iv_data

In [ ]:
# Plot the IV curve
fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.plot(iv_result['voltage'], iv_result['current'])
ax.set_xlabel('voltage / V')
ax.set_ylabel('current / A')
pass

## Shut down

Return to Voc mode and disable the output.

In [13]:
# Go to Voc
opet.mode = 'voc'

In [14]:
# Turn off output
opet.output_enabled = False